## 1. Load data

Load from the complete dataset in `data/raw/tennis_atp-master/` (main-tour singles). You can subset by **year range** or **max_rows** if the full set is too large. If no local data is found, we attempt to download a sample from Jeff Sackmann's GitHub.

In [1]:
import os
import gc
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss
import lightgbm as lgb
import joblib

# Add project root for src imports
sys.path.insert(0, str(Path().resolve()))
from src.data_loader import load_atp_matches
from src.preprocessing import clean_matches
from src.features import add_elo_features, add_rolling_features, add_h2h_features, build_diff_dataset, add_elo_spec_column, ELO_SPEC_ID

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# --- Config: subset the full dataset if needed ---
# Use last N years only, or cap rows for faster runs
YEAR_MIN = 2015          # None = no lower bound
YEAR_MAX = None          # None = no upper bound
MAX_ROWS = 100_000       # None = use all rows (can be slow for full history)

matches = load_atp_matches(raw_dir=RAW_DIR, tennis_atp_subdir='tennis_atp-master',
                           year_min=YEAR_MIN, year_max=YEAR_MAX, max_rows=MAX_ROWS)
if len(matches) == 0:
    url = 'https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_2024.csv'
    print('No local CSVs found — downloading sample from GitHub:', url)
    matches = pd.read_csv(url, low_memory=False)
    matches.to_csv(RAW_DIR / 'atp_matches_2024.csv', index=False)
    print('Downloaded to', RAW_DIR / 'atp_matches_2024.csv')
matches = clean_matches(matches)
print('Loaded and cleaned rows:', len(matches))
matches.head()


Loaded and cleaned rows: 27672


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2015-339,Brisbane,hard,28,A,2015-01-04,1,105357,NaN,WC,...,31.0,20.0,5.0,8.0,1.0,5.0,153.0,328.0,220.0,221.0
1,2015-339,Brisbane,hard,28,A,2015-01-04,22,105777,4.0,NaN,...,37.0,24.0,5.0,9.0,1.0,5.0,11.0,3645.0,34.0,1094.0
2,2015-339,Brisbane,hard,28,A,2015-01-04,9,105062,NaN,NaN,...,56.0,46.0,15.0,15.0,3.0,6.0,69.0,705.0,201.0,242.0
3,2015-339,Brisbane,hard,28,A,2015-01-04,10,106423,NaN,WC,...,33.0,23.0,7.0,9.0,1.0,4.0,149.0,341.0,25.0,1365.0
4,2015-339,Brisbane,hard,28,A,2015-01-04,7,103997,NaN,Q,...,43.0,30.0,17.0,11.0,4.0,7.0,177.0,282.0,16.0,2080.0


## 2. Minimal cleaning & standardization

We'll keep only essential columns for the demo and ensure date and surface are normalized.

In [2]:
# Cleaning (column canonicalization, dates, surface, ranks) was done in the load step above.
print('Rows after load+clean:', len(matches))
matches.head()

Rows after load+clean: 27672


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2015-339,Brisbane,hard,28,A,2015-01-04,1,105357,NaN,WC,...,31.0,20.0,5.0,8.0,1.0,5.0,153.0,328.0,220.0,221.0
1,2015-339,Brisbane,hard,28,A,2015-01-04,22,105777,4.0,NaN,...,37.0,24.0,5.0,9.0,1.0,5.0,11.0,3645.0,34.0,1094.0
2,2015-339,Brisbane,hard,28,A,2015-01-04,9,105062,NaN,NaN,...,56.0,46.0,15.0,15.0,3.0,6.0,69.0,705.0,201.0,242.0
3,2015-339,Brisbane,hard,28,A,2015-01-04,10,106423,NaN,WC,...,33.0,23.0,7.0,9.0,1.0,4.0,149.0,341.0,25.0,1365.0
4,2015-339,Brisbane,hard,28,A,2015-01-04,7,103997,NaN,Q,...,43.0,30.0,17.0,11.0,4.0,7.0,177.0,282.0,16.0,2080.0


## 3. Elo implementation (online)

We will compute an Elo rating per player using chronological updates. We'll keep a global Elo and a surface-specific Elo (hard/clay/grass)

In [3]:
matches = add_elo_features(matches)
print('Elo computed for matches')

Elo computed for matches


In [4]:
matches['elo_surf_before_w']

0        1500.000000
1        1500.000000
2        1500.000000
3        1500.000000
4        1500.000000
            ...     
27667    1689.724819
27668    1515.034373
27669    1653.741957
27670    1499.930689
27671    1538.009915
Name: elo_surf_before_w, Length: 27672, dtype: float64

## 4. Rolling features

form (last K matches win rate), surface winrate, days_since_last_match

We'll build a per-player running history and then for each match extract player-level stats *up to but not including that match*.

In [5]:
# Rolling features (form, surface form, days since last) via src.features
K_FORM = 8
matches = add_rolling_features(matches, K=K_FORM)
print('Rolling features added')

Rolling features added


In [6]:
matches['form_w']

0        1.000
1        0.500
2        0.000
3        0.000
4        0.000
         ...  
27667    0.625
27668    0.625
27669    0.750
27670    0.375
27671    0.625
Name: form_w, Length: 27672, dtype: float64

## 4b. Head-to-head features (Bayesian smoothing)

Add H2H win rate for winner and loser vs each other (past matches only). Smoothed with a prior so few H2H meetings don't dominate.

In [7]:
matches = add_h2h_features(matches, prior_weight=3.0, prior_win_rate=0.5)
matches = add_elo_spec_column(matches, ELO_SPEC_ID)
print('H2H features added; elo_spec_id =', ELO_SPEC_ID)

H2H features added; elo_spec_id = online_K32_base1500_surface_v1


## 5. Build match-difference dataset

We'll create a single row per match where features are differences (winner - loser) and label=1. We'll also create a mirrored dataset (loser-winner, label=0) for model training stability.

In [8]:
# Build difference dataset (includes H2H when present) via src.features
df_model, features = build_diff_dataset(matches, include_h2h=True)
LGB_CATEGORICAL_COLS = [c for c in ('player_a_id', 'player_b_id') if c in features]
features_logistic = [c for c in features if c not in LGB_CATEGORICAL_COLS]
X = df_model[features]
y = df_model['label'].astype(int)
print('Model dataset rows:', len(df_model))
print('Features:', features)
print('LightGBM categoricals:', LGB_CATEGORICAL_COLS)
print('Logistic (numeric only):', features_logistic)

Model dataset rows: 55344
Features: ['elo_diff', 'elo_surf_diff', 'form_diff', 'surf_form_diff', 'days_since_diff', 'is_hard', 'is_clay', 'is_grass', 'elo_a', 'elo_b', 'elo_surf_a', 'elo_surf_b', 'player_a_id', 'player_b_id', 'rank_diff', 'h2h_diff']
LightGBM categoricals: ['player_a_id', 'player_b_id']
Logistic (numeric only): ['elo_diff', 'elo_surf_diff', 'form_diff', 'surf_form_diff', 'days_since_diff', 'is_hard', 'is_clay', 'is_grass', 'elo_a', 'elo_b', 'elo_surf_a', 'elo_surf_b', 'rank_diff', 'h2h_diff']


In [9]:
y

0        1
1        0
2        1
3        0
4        1
        ..
55339    0
55340    1
55341    0
55342    1
55343    0
Name: label, Length: 55344, dtype: int64

## 6. Time-aware train/test split and baseline models

We'll use TimeSeriesSplit to keep temporal order. For simplicity we'll do a single train/validation/test split based on date index.

In [10]:
# Find a date split: train on first 70%, val 15%, test 15% by time
n = len(matches)
train_date = matches['tourney_date'].iloc[int(0.7*n)]
val_date = matches['tourney_date'].iloc[int(0.85*n)]

# To map df_model rows back to dates we can reconstruct indices: note we made two rows per match in chronological order, so groups of 2 correspond to same match order
# We'll assign a "match_idx" column
num_matches = len(matches)
match_indices = np.repeat(np.arange(num_matches), 2)
if len(match_indices) != len(df_model):
    # safeguard: align by truncation or repetition
    match_indices = match_indices[:len(df_model)]

# get match dates for df_model
match_dates = matches['tourney_date'].iloc[match_indices].reset_index(drop=True)

# create boolean masks
train_mask = match_dates <= train_date
val_mask = (match_dates > train_date) & (match_dates <= val_date)
test_mask = match_dates > val_date

X_train, y_train = X[train_mask].copy(), y[train_mask].copy()
X_val, y_val = X[val_mask].copy(), y[val_mask].copy()
X_test, y_test = X[test_mask].copy(), y[test_mask].copy()

print('Train/Val/Test sizes:', len(X_train), len(X_val), len(X_test))

# %%
# Baseline: logistic regression (numeric features only; player ids are for tree / mixed-effect-style models)
X_train_log = X_train[features_logistic]
X_val_log = X_val[features_logistic]
X_test_log = X_test[features_logistic]
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_log)
X_val_s = scaler.transform(X_val_log)
X_test_s = scaler.transform(X_test_log)

log = LogisticRegression(max_iter=1000)
log.fit(X_train_s, y_train)

def evaluate_model(clf, Xs, ys, name='model'):
    preds = clf.predict(Xs)
    probs = clf.predict_proba(Xs)[:,1] if hasattr(clf, 'predict_proba') else clf.decision_function(Xs)
    acc = accuracy_score(ys, preds)
    auc = roc_auc_score(ys, probs)
    brier = brier_score_loss(ys, probs)
    print(f'{name} — acc: {acc:.4f}, auc: {auc:.4f}, brier: {brier:.4f}')
    return {'acc':acc, 'auc':auc, 'brier':brier}

print('Logistic performance')
_eval_train = evaluate_model(log, X_train_s, y_train, 'Logistic (train)')
_eval_val = evaluate_model(log, X_val_s, y_val, 'Logistic (val)')
_eval_test = evaluate_model(log, X_test_s, y_test, 'Logistic (test)')

Train/Val/Test sizes: 38820 8368 8156


Logistic performance
Logistic (train) — acc: 0.7522, auc: 0.8292, brier: 0.1704
Logistic (val) — acc: 0.7450, auc: 0.8313, brier: 0.1695
Logistic (test) — acc: 0.7401, auc: 0.8148, brier: 0.1772


## 7. LightGBM classifier (non-lambdarank)

Difference features plus match-level pre-match Elo (`elo_a`/`elo_b`, `elo_surf_*`) and `player_a_id`/`player_b_id` as **categorical** features (tree-based surrogate for player heterogeneity; not a formal GLMM). Section **7c** in the training cell prints a repeated-measures diagnostic.

In [11]:
lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=LGB_CATEGORICAL_COLS)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=LGB_CATEGORICAL_COLS)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_data_in_leaf': 50,
    'max_depth': 10,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'seed': RANDOM_SEED,
}

bst = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train','val'],
    num_boost_round=2000,
    #early_stopping_rounds=100,
    #verbose_eval=100,
)


def summarize_preds(name, y_true, probs):
    preds = (probs > 0.5).astype(int)
    acc = accuracy_score(y_true, preds)
    auc = roc_auc_score(y_true, probs)
    brier = brier_score_loss(y_true, probs)
    print(f'LightGBM ({name}) — acc: {acc:.4f}, auc: {auc:.4f}, brier: {brier:.4f}')


preds_val = bst.predict(X_val, num_iteration=bst.best_iteration)
preds_test = bst.predict(X_test, num_iteration=bst.best_iteration)
summarize_preds('val', y_val, preds_val)
summarize_preds('test', y_test, preds_test)

# 7c. Repeated measures: rows are not independent (mirrored pairs + same players across time)
rows_test = df_model.loc[test_mask].copy()
rows_test['prob'] = preds_test
rows_test['brier'] = (rows_test['prob'] - rows_test['label']) ** 2
if 'player_a_id' in rows_test.columns:
    by_pid = rows_test.groupby('player_a_id', observed=True)['brier'].agg(['mean', 'count'])
    print('Repeated-measures diagnostic (focal player_a): distribution of per-player mean test Brier')
    print(by_pid['mean'].describe())
print('Note: primary evaluation uses chronological split; classical iid confidence intervals understate uncertainty here.')

[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero


LightGBM (val) — acc: 0.8470, auc: 0.9337, brier: 0.1152
LightGBM (test) — acc: 0.8254, auc: 0.9120, brier: 0.1357
Repeated-measures diagnostic (focal player_a): distribution of per-player mean test Brier
count    4.880000e+02
mean     1.289316e-01
std      1.545795e-01
min      2.246422e-10
25%      5.375929e-03
50%      8.597305e-02
75%      1.782903e-01
max      9.920644e-01
Name: mean, dtype: float64
Note: primary evaluation uses chronological split; classical iid confidence intervals understate uncertainty here.


## 7b. Optuna hyperparameter tuning (optional)

Tune LightGBM classifier hyperparameters on the validation set (minimize Brier or maximize AUC). Uses the same time-based train/val split.

In [12]:
import optuna
from optuna.integration import LightGBMPruningCallback

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': 1,
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        'seed': RANDOM_SEED,
    }
    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=LGB_CATEGORICAL_COLS)
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=LGB_CATEGORICAL_COLS)
    callbacks = [LightGBMPruningCallback(trial, 'auc', 'val'), lgb.early_stopping(80, verbose=False)]
    model = lgb.train(
        params, lgb_train,
        valid_sets=[lgb_val], valid_names=['val'],
        num_boost_round=1500, callbacks=callbacks,
    )
    preds = model.predict(X_val, num_iteration=model.best_iteration)
    return roc_auc_score(y_val, preds)  # maximize AUC (matches LightGBMPruningCallback)

# Run a short study (increase n_trials for better results)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED, n_startup_trials=5))
study.optimize(objective, n_trials=15, show_progress_bar=True)
print('Best trial AUC:', study.best_value)
print('Best params:', study.best_params)
# Optionally retrain with best params and save as bst

/home/tvm0867/Hobby_projects/tennis_prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-10 11:02:57,894] A new study created in memory with name: no-name-97ce3887-9541-4016-b5e2-efdfcaae383a



  0%|                                                                       | 0/15 [00:00<?, ?it/s]

[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero



  0%|                                                                       | 0/15 [00:01<?, ?it/s]


Best trial: 0. Best value: 0.945736:   0%|                                  | 0/15 [00:01<?, ?it/s]

...


Best trial: 0. Best value: 0.945736:  93%|███████████████████████▎ | 14/15 [00:10<00:00,  2.52it/s]


Best trial: 0. Best value: 0.945736:  93%|███████████████████████▎ | 14/15 [00:10<00:00,  2.52it/s]


Best trial: 0. Best value: 0.945736: 100%|█████████████████████████| 15/15 [00:10<00:00,  3.64it/s]


Best trial: 0. Best value: 0.945736: 100%|█████████████████████████| 15/15 [00:10<00:00,  1.43it/s]

[I 2026-05-10 11:03:08,261] Trial 12 pruned. Trial was pruned at iteration 0.
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[I 2026-05-10 11:03:08,337] Trial 13 pruned. Trial was pruned at iteration 0.
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[I 2026-05-10 11:03:08,407] Trial 14 pruned. Trial was pruned at iteration 0.
Best trial AUC: 0.9457358154893998
Best params: {'learning_rate': 0.0425378509573798, 'num_leaves': 61, 'min_data_in_leaf': 76, 'feature_fraction': 0.8394633936788146, 'bagging_fraction': 0.6624074561769746, 'lambd

## 8. LambdaRank approach (learning-to-rank)

This uses two rows per match but with player features instead of differences and trains objective='lambdarank'.

## 8b. SHAP explainability

TreeExplainer on the LightGBM classifier: summary plot and mean |SHAP| per feature (sample of validation set for speed).

In [13]:
import matplotlib.pyplot as plt
import shap

# Use the trained LightGBM classifier (bst) and a sample of validation data
sample_size = min(500, len(X_val))
X_val_sample = X_val.sample(n=sample_size, random_state=RANDOM_SEED) if hasattr(X_val, 'sample') else X_val.iloc[:sample_size]
explainer = shap.TreeExplainer(bst, feature_names=features)
shap_values = explainer.shap_values(X_val_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # positive class for binary

# Summary plot (beeswarm)
shap.summary_plot(shap_values, X_val_sample, feature_names=features, show=False)
plt.gcf().set_size_inches(8, 6)
plt.title('SHAP summary (val sample)')
plt.tight_layout()
plt.show()

# Bar plot: mean |SHAP| per feature
shap.summary_plot(shap_values, X_val_sample, feature_names=features, plot_type='bar', show=False)
plt.gcf().set_size_inches(8, 5)
plt.title('Mean |SHAP| per feature')
plt.tight_layout()
plt.show()

/home/tvm0867/Hobby_projects/tennis_prediction/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/tmp/ipykernel_45093/1462984213.py:13: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_val_sample, feature_names=features, show=False)
/tmp/ipykernel_45093/1462984213.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_45093/1462984213.py:20: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_

/tmp/ipykernel_45093/1462984213.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
def _player_row(row, role, match_idx):
    prefix = 'w' if role == 'w' else 'l'
    rank_key = 'winner_rank' if role == 'w' else 'loser_rank'
    pid = int(row['winner_id']) if role == 'w' else int(row['loser_id'])
    return {
        'match_id': match_idx,
        'player_id': pid,
        'elo_before': row[f'elo_before_{prefix}'],
        'elo_surf_before': row[f'elo_surf_before_{prefix}'],
        'form': row[f'form_{prefix}'],
        'surf_form': row[f'surf_form_{prefix}'],
        'days_since': row[f'days_since_{prefix}'],
        'rank': row.get(rank_key, np.nan),
        'is_hard': 1 if row['surface'] == 'hard' else 0,
        'is_clay': 1 if row['surface'] == 'clay' else 0,
        'is_grass': 1 if row['surface'] == 'grass' else 0,
        'label': 1 if role == 'w' else 0
    }

player_rows = []
for match_idx, row in matches.iterrows():
    player_rows.append(_player_row(row, 'w', match_idx))
    player_rows.append(_player_row(row, 'l', match_idx))

rank_df = pd.DataFrame(player_rows)
rank_df['days_since'] = rank_df['days_since'].fillna(0.0)
rank_df['rank'] = rank_df['rank'].fillna(rank_df['rank'].median())
RANK_CATEGORICAL_COLS = ['player_id']
rank_features = ['elo_before','elo_surf_before','form','surf_form','days_since','rank','is_hard','is_clay','is_grass','player_id']
X_rank = rank_df[rank_features]
y_rank = rank_df['label'].astype(int)

match_train_mask = matches['tourney_date'] <= train_date
match_val_mask = (matches['tourney_date'] > train_date) & (matches['tourney_date'] <= val_date)
match_test_mask = matches['tourney_date'] > val_date

train_rank_mask = np.repeat(match_train_mask.values, 2)
val_rank_mask = np.repeat(match_val_mask.values, 2)
test_rank_mask = np.repeat(match_test_mask.values, 2)

rank_train = lgb.Dataset(X_rank[train_rank_mask], label=y_rank[train_rank_mask], group=[2]*match_train_mask.sum(), categorical_feature=RANK_CATEGORICAL_COLS)
rank_val = lgb.Dataset(X_rank[val_rank_mask], label=y_rank[val_rank_mask], group=[2]*match_val_mask.sum(), reference=rank_train, categorical_feature=RANK_CATEGORICAL_COLS)

rank_params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [2],
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_data_in_leaf': 30,
    'seed': RANDOM_SEED,
}

ranker = lgb.train(
    rank_params,
    rank_train,
    valid_sets=[rank_train, rank_val],
    valid_names=['train','val'],
    num_boost_round=2000,
    #early_stopping_rounds=100,
    #verbose_eval=100,
)


def evaluate_rank_split(model, mask, split_name):
    pair_mask = np.repeat(mask.values, 2)
    scores = model.predict(X_rank[pair_mask], num_iteration=model.best_iteration)
    labels = y_rank[pair_mask].values
    match_scores = scores.reshape(-1, 2)
    winner_scores = match_scores[:, 0]
    loser_scores = match_scores[:, 1]
    probs = 1 / (1 + np.exp(loser_scores - winner_scores))
    acc = (winner_scores > loser_scores).mean()
    auc = roc_auc_score(labels, scores)
    brier = np.mean((probs - labels.reshape(-1, 2)[:, 0]) ** 2)
    print(f'LambdaRank ({split_name}) — match acc: {acc:.4f}, pairwise auc: {auc:.4f}, brier: {brier:.4f}')
    return {'acc': acc, 'auc': auc, 'brier': brier}

rank_metrics = {
    'train': evaluate_rank_split(ranker, match_train_mask, 'train'),
    'val': evaluate_rank_split(ranker, match_val_mask, 'val'),
    'test': evaluate_rank_split(ranker, match_test_mask, 'test'),
}

[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero


LambdaRank (train) — match acc: 0.9997, pairwise auc: 0.8705, brier: 0.0003


LambdaRank (val) — match acc: 0.8444, pairwise auc: 0.7935, brier: 0.1196


LambdaRank (test) — match acc: 0.8203, pairwise auc: 0.7820, brier: 0.1380


## 9. Calibration check and simple reliability diagram

In [15]:
import matplotlib.pyplot as plt

probs = preds_val
bins = np.linspace(0, 1, 11)
bin_idx = np.digitize(probs, bins) - 1
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_true = []
for i in range(len(bin_centers)):
    mask = bin_idx == i
    bin_true.append(y_val[mask].mean() if mask.any() else np.nan)

plt.figure(figsize=(6,5))
plt.plot(bin_centers, bin_true, marker='o')
plt.plot([0,1],[0,1],'--', alpha=0.5)
plt.xlabel('Predicted probability')
plt.ylabel('Observed frequency')
plt.title('Reliability diagram (val set)')
plt.show()

### Appendix: formal GLMM (optional)

For a **variance component** for players (beyond LightGBM categoricals), fit a **binary GLMM**—e.g. R `lme4::glmer`, Python **pymer4**, or **Bambi**—usually on **one row per match** with fixed effects for Elo diff and surface; random intercepts for both sides need a careful Bradley–Terry-style parametrization.

---

## 10. Save models and scalers

In [16]:
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(scaler, MODEL_DIR / 'scaler.joblib')
joblib.dump(log, MODEL_DIR / 'logistic.joblib')
joblib.dump(features_logistic, MODEL_DIR / 'logistic_features.joblib')
joblib.dump(features, MODEL_DIR / 'lgb_classifier_features.joblib')
bst.save_model(str(MODEL_DIR / 'lgb_model.txt'), num_iteration=bst.best_iteration)
joblib.dump(rank_features, MODEL_DIR / 'rank_features.joblib')
ranker.save_model(str(MODEL_DIR / 'lgb_lambdarank.txt'), num_iteration=ranker.best_iteration)
print('Saved models to', MODEL_DIR)

Saved models to models
